# A pretrained MobileNetV2: full and selected CIM conversion
Load torchvision's ImageNet model directly, evaluate it, then compare all-layer CIM with CIM that keeps the first convolution and final classifier digital. Supply the ImageNet validation directory through `MEMFAST_IMAGENET_ROOT`. Run cells in order after installing MemFast. Pretrained weights use torchvision's upstream download/cache.

In [ ]:
import os
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models

from memfast import Config, Nonidealities, Simulator, convert_model

torch.manual_seed(135)
torch.set_num_threads(4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
data_root = Path(os.environ.get("MEMFAST_IMAGENET_ROOT", "data/imagenet/val"))
max_samples = 128  # Set None to evaluate the full validation split.
batch_size = 8
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights).to(device).eval()
print("Pretrained model:", type(model).__name__, "device:", device)

## 1. Load the evaluation split with pretrained preprocessing
Class directories must follow ImageNet's standard 1,000-class index order. A subset is a workflow comparison, not full ImageNet accuracy.

In [ ]:
dataset = datasets.ImageFolder(data_root, transform=weights.transforms())
if len(dataset.classes) != 1000:
    raise ValueError("Expected all 1,000 ImageNet class directories in standard order")
selected = (
    dataset if max_samples is None else Subset(dataset, range(min(max_samples, len(dataset))))
)
batches = DataLoader(selected, batch_size=batch_size, shuffle=False)


@torch.inference_mode()
def accuracy(network):
    correct = count = 0
    network.eval()
    for images, labels in batches:
        logits = network(images.to(device))
        assert torch.isfinite(logits).all()
        correct += int((logits.argmax(1).cpu() == labels).sum())
        count += labels.numel()
    if count == 0:
        raise ValueError("No validation images found")
    return correct / count, count


digital, count = accuracy(model)
print(f"Digital top-1: {digital:.2%}, {count} images")

## 2. Convert every supported layer
Conv2d (including depthwise) and Linear become CIM layers. Batch normalization, activations, pooling and residual additions retain their PyTorch behavior. Only write variation is enabled at 0.02.

In [ ]:
config = Config(nonidealities=Nonidealities(write_variation=0.02), seed=135)
engine = Simulator(config, device=device)
analog = convert_model(model, engine)
full_cim, _ = accuracy(analog)
print(f"Full CIM top-1: {full_cim:.2%}, {count} images")
del analog, engine

## 3. Keep the first convolution and classification head digital
Inspect module names, then pass exclusions directly. Each conversion starts from the same pretrained digital model.

In [ ]:
eligible = [
    (name, type(layer).__name__)
    for name, layer in model.named_modules()
    if isinstance(layer, (nn.Conv2d, nn.Linear))
]
print("First:", eligible[0], "last:", eligible[-1])
first_layer = eligible[0][0]
head = "classifier"
engine = Simulator(config, device=device)
partial = convert_model(model, engine, exclude=(first_layer, head))
assert isinstance(partial.get_submodule(first_layer), nn.Conv2d)
assert isinstance(partial.classifier[1], nn.Linear)
partial_cim, _ = accuracy(partial)
print(f"Partial CIM top-1: {partial_cim:.2%}, {count} images")
comparison = {"digital": digital, "full CIM": full_cim, "first/head digital": partial_cim}
comparison

Accuracy differences depend on checkpoint, dataset, quantization and device parameters. This comparison uses identical images and a fixed programming realization; no improvement from excluding layers is assumed. Use `include=(...)` to select only named layers instead.